<a href="https://colab.research.google.com/github/Isobit7/SETU/blob/main/notebooks/SETU_Cloud_Training_STGCN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SETU — Cloud High-Accuracy Sign Language Training Pipeline
## Train Indian Sign Language (ISL) Multi-Signer Neural Models on Cloud GPU (Colab / Kaggle)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Isobit7/SETU/blob/main/notebooks/SETU_Cloud_Training_STGCN.ipynb)

This notebook trains a **Deep Residual Multi-Signer Model (126-D Dual-Hand)** with 3D spatial landmark augmentations and exports both PyTorch (`.pth`) and ONNX (`.onnx`) weights directly for use in the SETU web platform.

In [1]:
# 1. Verify Cloud GPU Acceleration
!nvidia-smi

# Install requirements
!pip install -q torch torchvision mediapipe numpy onnx onnxruntime matplotlib pytest


In [2]:
# 2. Safely Clone or Update SETU Codebase
import os, sys

# Clean up any accidental nested directories
if os.path.exists('/content/SETU/SETU'):
    !rm -rf /content/SETU/SETU

if not os.path.exists('/content/SETU/.git'):
    !rm -rf /content/SETU
    !git clone https://github.com/Isobit7/SETU.git /content/SETU
else:
    %cd /content/SETU
    !git pull origin main

%cd /content/SETU

# Configure PYTHONPATH so all modules (training, scripts, backend) resolve cleanly
os.environ['PYTHONPATH'] = '/content/SETU:' + os.environ.get('PYTHONPATH', '')
if '/content/SETU' not in sys.path:
    sys.path.insert(0, '/content/SETU')
print('SETU Root:', os.getcwd())


In [3]:
# 3. Run High-Accuracy Multi-Signer Training (40 Epochs with Cosine Annealing)
!PYTHONPATH=. python training/train_cloud_high_accuracy.py --epochs 40 --batch-size 64 --lr 0.001


In [4]:
# 4. Validate Model Parity & Test Accuracy (11 Comprehensive Tests)
!PYTHONPATH=. python -m pytest tests/test_model_accuracy_benchmarks.py tests/test_onnx_edge_and_conversation.py -v


In [5]:
# 5. Download Trained Model Weights (.pth and .onnx) for SETU Deployment
from google.colab import files
files.download('models/weights/realsign_multisigner_mlp.pth')
files.download('models/weights/realsign_26class_mlp.pth')
files.download('models/weights/realsign_multisigner.onnx')
